# LINT Pipeline

## Import

In [99]:
from lint_ii import ReadabilityAnalysis
import pandas as pd
import numpy as np

from anthropic import Anthropic
from openai import OpenAI
from google import genai
from google.genai import types
import lmstudio as lms
from pydantic import BaseModel

from dotenv import load_dotenv
import os

## Rewrite Engines, Response schemas

### Engine

In [60]:
class RewriteEngine:
     """
     Engine that rewrites a text using a specific LLM
     
     
     """
     def __init__(self, 
                  model:str, 
                  name:str, 
                  is_dutch:bool = False, 
                  is_local:bool = False, 
                  is_open_source:bool = False, 
                  is_large:bool = False):
          
          self.model = model
          self.name = name
          self.is_dutch = is_dutch
          self.is_local = is_local
          self.is_open_source = is_open_source
          self.is_large = is_large
          pass
     
     def set_instruction(self, instruction:str):
          self.instruction = instruction
     
     def prompt_model(self, user_prompt:str):
          pass

In [61]:
# Response schema
class RewriteResponse(BaseModel):
    original_sentence: str
    rewritten_sentences: str

class RewrittenText(BaseModel):
    text: list[RewriteResponse]

In [62]:
# Specifically for Claude
response_schema = {
    "format": {
        "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "original_sentence": {
                                    "type": "string"
                                },
                                "rewritten_sentences": {
                                    "type": "string"
                                }
                            },
                            "required": [
                                "original_sentence",
                                "rewritten_sentences"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                "required": ["text"],
                "additionalProperties": False
            },
        
        }
    }

### LMStudio

In [63]:
# LMStudio hosts and manages all my local models. It optimizes gpu allocation so i don't have to.
class RE_LMStudio(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = lms.llm(self.model) # 'fietje-2-instruct'

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        chat = lms.Chat(self.instruction)
        chat.add_user_message(user_prompt)
        
        response = self.client.respond(
            chat,
            config={
                "temperature": 1.0,
                # "maxTokens": 50,
            },
            response_format=RewrittenText,
        )
        return response.parsed

### Claude

In [78]:
class RE_Claude(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        response = self.client.messages.parse(
            model=self.model, # "claude-opus-4-7"
            output_format=RewrittenText,
            system=self.instruction,
            max_tokens=1024,
            messages=[
                {
                    "role": "user",
                    "content": f"{user_prompt}",
                }
            ],
        )
        return response.parsed_output

### OpenAI

In [65]:
class RE_OpenAi(RewriteEngine):

    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)

    def prompt_model(self, user_prompt) -> str:
        response = self.client.responses.parse(
            model=self.model, # "gpt-4.1-nano"
            instructions=self.instruction,
            text_format = RewrittenText,
            input=[
                {
                    "role": "user",
                    "content": f"{user_prompt}",
                }
            ],
            reasoning={},
            tools=[],
            temperature=0.5,
            max_output_tokens=10000,
            top_p=1,
            store=True,
            include=["web_search_call.action.sources"]
            )
        return response.output_parsed

### Gemini

In [66]:
class RE_Gemini(RewriteEngine):


    def __init__(self, model, name, is_dutch = False, is_local = False, is_open_source = False, is_large = False):
        super().__init__(model, name, is_dutch, is_local, is_open_source, is_large)
        self.client = genai.Client()

    def set_instruction(self, instruction):
        return super().set_instruction(instruction)
    
    def prompt_model(self, user_prompt):
        response = self.client.models.generate_content(
            model=self.model, # "gemini-3-flash-preview" 
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="low"),
                system_instruction=self.instruction,
                response_mime_type="application/json",
                response_schema=RewrittenText
            ),
            contents=f"{user_prompt}"
        )

        rewritten = RewrittenText.model_validate_json(response.text)
        return rewritten

## LINT Pipeline

In [125]:
class LintPipe:

    def __init__(self, instruction_en:str, instruction_nl:str, user_prompt:str):
        self.user_prompt = user_prompt
        self.analysis = self.get_lint_analysis(self.user_prompt)
        
        self.instruction_en = self.format_instruction(instruction_en)
        self.instruction_nl = self.format_instruction(instruction_nl, is_dutch = True)
        
        self.rewriteEngines = []
        self.prompt_responses = []
        return

    def format_instruction(self, instruction:str, is_dutch:bool = False):
        # Determine document and sentence scores:
        sent_scores = [int(sent_analysis['score']) for sent_analysis in self.analysis['sentence_stats']]
        doc_score = str(int(self.analysis['document_stats']['document_lint_score']))
        
        if is_dutch:
            sent_scores = "{} en {}".format(", ".join(map(str, sent_scores[:-1])),  str(object=sent_scores[-1]))
        else:
            sent_scores = "{} and {}".format(", ".join(map(str, sent_scores[:-1])),  str(sent_scores[-1]))

        instruction = instruction.replace('<doc_score>', doc_score)
        instruction = instruction.replace('<sent_scores>', sent_scores)

        return instruction


    def add_engine(self, engine:RewriteEngine):
        # TODO: Retrieve all top_n uncommon 

        if engine.is_dutch:
            engine.set_instruction(self.instruction_nl)
        else:
            engine.set_instruction(self.instruction_en)
        self.rewriteEngines.append(engine)

    def list_engines(self):
        print("Rewrite engines:")
        for engine in self.rewriteEngines:
            print(f"NAME: {engine.name}, MODEL: '{engine.model}', ?NL {engine.is_dutch}, ?LOCAL: {engine.is_local}, ?OS {engine.is_open_source}, ?LARGE {engine.is_large}")

    def prompt_engines(self):
        # TODO: Prompt engines moet doen:
        # 1. Return een clean object:
        #     - Model id
        #     - Lint scores per zinsblok.
        #     - Herschreven tekst

        # - Dit object moet ook een methode hebben: extract_text en get_overall_lint_score()
        self.prompt_responses = []
        results = []
        if [] == self.rewriteEngines:
            print("No Rewrite Engines added. Use `add_engine` to do so.")
            return

        for engine in self.rewriteEngines:
            print(f"Rewriting using {engine.name} - Model: {engine.model}")
            engine_id = f'{engine.name}/{engine.model}'
            try:
                engine_result = engine.prompt_model(self.user_prompt)
                print(engine_result)
                if type(engine_result) == RewrittenText:
                    print("is rewritten object")
                    engine_result = engine_result.model_dump()
                full_text = " ".join([rs['rewritten_sentences'] for rs in engine_result['text']])
                engine_result['full_text'] = full_text
                engine_result['engine'] = engine_id
                results.append(engine_result)
            except Exception as e:
                print(f"Something went wrong with {engine_id}, skipping...")
                print(e)
                faulty_engine = dict()
                faulty_engine['engine'] = engine_id
                faulty_engine['full_text'] = ''
                results.append(faulty_engine)
        
        self.prompt_responses = results.copy()
        return results

    def get_lint_analysis(self, user_prompt):
        analysis = ReadabilityAnalysis.from_text(user_prompt)
        return analysis.get_detailed_analysis()
    
    def eval_response(self):
        response_object = {}
        
        for resp in self.prompt_responses:
            print("bezig met", resp['engine'])
            analysis = ReadabilityAnalysis.from_text(resp['full_text'])
            doc_score = sent_scores = ""
            if analysis.lint.score == None:
                doc_score = sent_scores = np.nan
            else: 
                doc_score = str(int(analysis.lint.score))
                sent_scores = map(int, analysis.lint_scores_per_sentence)
                sent_scores = ", ".join(map(str, sent_scores))
            
            tr = {f"{resp['engine']} - text": resp['full_text'], f"{resp['engine']} - lint" : doc_score, f"{resp['engine']} - sent": sent_scores}
            print(tr)
            response_object.update(tr)

        return pd.DataFrame([response_object])

        


## Instructions

In [10]:
instruction_en = """"
Goal: Your goal is to minimize the readability score of a text. The lower the score the better.

The readability score is calculated as follows: 100 - (
      - 4.21
      + (17.28 * word frequency)
      - (1.62  * syntactic dependency length)
      - (2.54  * content words per clause)
      + (16.00 * proportion concrete nouns)
  )

The current readability score of the text is <doc_score>, the goal is 34 or lower. The current scores per sentence are <sent_scores>.  

Method:
- Do not rewrite sentences with a readability score lower than 34.
- Do not add information that wasn't there, leave the sentences as much in tact as possible to complete the task.
- Reduce words with low frequency use by finding synonyms or rewriting the sentence.
- Lower the syntactic dependency length by rewriting the sentence.
- Lower the content words per clause by splitting long sentences in smaller ones.
- Increase the proportion of concrete nouns by minimizing abstract nouns in the text.
- Perform this method per sentence.
- Format your response in plain text only. Do not use LaTeX, MathJax, or any markup notation such as \( \), $, or \frac{}{}. Write all math expressions using standard text characters (e.g., "/" for division, "*" for multiplication, and "^" for exponents).

"""

In [11]:
instruction_nl = """
Doel: Het doel is om de leesbaarheidsscore van een tekst zo laag mogelijk te maken. Hoe lager de score, hoe beter.

De leesbaarheidsscore wordt als volgt berekend: 100 - (
        - 4.21
        + (17.28 * woordfrequentie)
        - (1.62 * syntactische afhankelijkheidslengte)
        - (2.54 * inhoudswoorden per bijzin)
        + (16.00 * aandeel concrete zelfstandige naamwoorden)
    )

De huidige leesbaarheidsscore van de tekst is <doc_score>; het doel is 34 of lager. De huidige scores per zin zijn <sent_scores>.

Methode:
- Herschrijf geen zinnen met een leesbaarheidsscore lager dan 34.
- Voeg geen informatie toe die er niet stond; laat de zinnen zoveel mogelijk intact om de taak uit te voeren.
- Verminder het gebruik van woorden met een lage frequentie door synoniemen te gebruiken of de zin te herschrijven.
- Verlaag de syntactische afhankelijkheidslengte door de zin te herschrijven.
- Verlaag het aantal inhoudswoorden per bijzin door lange zinnen op te splitsen in kortere zinnen.
- Verhoog het aandeel concrete zelfstandige naamwoorden door abstracte zelfstandige naamwoorden zoveel mogelijk te vermijden.
- Pas deze methode per zin toe.
- Formatteer je antwoord uitsluitend als platte tekst. Gebruik geen LaTeX, MathJax of andere opmaaknotatie zoals \( \), $, of \frac{}{}. Schrijf alle wiskundige uitdrukkingen met standaardteksttekens (bijvoorbeeld "/" voor deling, "*" voor vermenigvuldiging en "^" voor exponenten).

"""

In [12]:
inp_txt = """De Oudegracht is het sfeervolle hart van de stad.
In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. 
Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen."""

In [102]:
chatgpt = RE_OpenAi(
    model="gpt-4.1-nano",
    name = "ChatGPT",
    is_local=False,
    is_open_source=False,
    is_large=True
)

gemini = RE_Gemini(
    model="gemini-3-flash-preview",
    name="Gemini",
    is_local=False,
    is_open_source=False,
    is_large=True
)

claude = RE_Claude(
    model='claude-sonnet-4-6',
    name="Claude",
    is_local=False,
    is_open_source=False,
    is_large=False
)

fietje = RE_LMStudio(
    model="fietje-2-instruct",
    name="Fietje",
    is_dutch=True,
    is_local=True,
    is_open_source=True,
    is_large=False
)

In [126]:
test_pipe = LintPipe(instruction_en, instruction_nl, inp_txt)
test_pipe.add_engine(chatgpt)
test_pipe.add_engine(gemini)
test_pipe.add_engine(claude)
test_pipe.add_engine(fietje)


In [127]:
test_pipe.list_engines()

Rewrite engines:
NAME: ChatGPT, MODEL: 'gpt-4.1-nano', ?NL False, ?LOCAL: False, ?OS False, ?LARGE True
NAME: Gemini, MODEL: 'gemini-3-flash-preview', ?NL False, ?LOCAL: False, ?OS False, ?LARGE True
NAME: Claude, MODEL: 'claude-sonnet-4-6', ?NL False, ?LOCAL: False, ?OS False, ?LARGE False
NAME: Fietje, MODEL: 'fietje-2-instruct', ?NL True, ?LOCAL: True, ?OS True, ?LARGE False


In [109]:
z = test_pipe.prompt_engines()

Rewriting using ChatGPT - Model: gpt-4.1-nano
text=[RewriteResponse(original_sentence='De Oudegracht is het sfeervolle hart van de stad.', rewritten_sentences='Oudegracht vormt het gezellige centrum van de stad.'), RewriteResponse(original_sentence='In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.', rewritten_sentences='In de middeleeuwen was het hier druk met handel en vervoer van goederen.'), RewriteResponse(original_sentence='Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.', rewritten_sentences='Tegenwoordig is het een mooie plek om te shoppen, te lunchen of te dineren in oude stadskastelen.')]
is rewritten object
Rewriting using Gemini - Model: gemini-3-flash-preview
text=[RewriteResponse(original_sentence='De Oudegracht is het sfeervolle hart van de stad.', rewritten_sentences='De Oudegracht is het sfeervolle hart van de stad.'), RewriteResponse(original_sentence='In de middeleeuwen was het 

In [128]:
z

[{'text': [{'original_sentence': 'De Oudegracht is het sfeervolle hart van de stad.',
    'rewritten_sentences': 'Oudegracht vormt het gezellige centrum van de stad.'},
   {'original_sentence': 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.',
    'rewritten_sentences': 'In de middeleeuwen was het hier druk met handel en vervoer van goederen.'},
   {'original_sentence': 'Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.',
    'rewritten_sentences': 'Tegenwoordig is het een mooie plek om te shoppen, te lunchen of te dineren in oude stadskastelen.'}],
  'full_text': 'Oudegracht vormt het gezellige centrum van de stad. In de middeleeuwen was het hier druk met handel en vervoer van goederen. Tegenwoordig is het een mooie plek om te shoppen, te lunchen of te dineren in oude stadskastelen.',
  'engine': 'ChatGPT/gpt-4.1-nano'},
 {'text': [{'original_sentence': 'De Oudegracht is het sfeervolle hart van 

In [129]:
test_pipe.prompt_responses = z

In [90]:
klm = test_pipe.prompt_responses[3]

In [91]:
analysis = ReadabilityAnalysis.from_text(klm['full_text'])

In [95]:
analysis.lint.score == None

True

In [130]:
test_pipe.eval_response()

bezig met ChatGPT/gpt-4.1-nano
{'ChatGPT/gpt-4.1-nano - text': 'Oudegracht vormt het gezellige centrum van de stad. In de middeleeuwen was het hier druk met handel en vervoer van goederen. Tegenwoordig is het een mooie plek om te shoppen, te lunchen of te dineren in oude stadskastelen.', 'ChatGPT/gpt-4.1-nano - lint': '47', 'ChatGPT/gpt-4.1-nano - sent': '38, 45, 61'}
bezig met Gemini/gemini-3-flash-preview
{'Gemini/gemini-3-flash-preview - text': 'De Oudegracht is het sfeervolle hart van de stad. Boten brachten vroeger spullen naar de kade. Het was daar erg druk. Nu kun je er winkelen. Je kunt eten in een oud kasteel.', 'Gemini/gemini-3-flash-preview - lint': '19', 'Gemini/gemini-3-flash-preview - sent': '18, 32, 12'}
bezig met Claude/claude-sonnet-4-6
{'Claude/claude-sonnet-4-6 - text': 'De Oudegracht is het sfeervolle hart van de stad. In de middeleeuwen was het hier druk. Boten brachten goederen aan en af over het water. Nu kan je er winkelen. Je kan er ook lunchen of dineren. De o

,ChatGPT/gpt-4.1-nano - text,ChatGPT/gpt-4.1-nano - lint,ChatGPT/gpt-4.1-nano - sent,Gemini/gemini-3-flash-preview - text,Gemini/gemini-3-flash-preview - lint,Gemini/gemini-3-flash-preview - sent,Claude/claude-sonnet-4-6 - text,Claude/claude-sonnet-4-6 - lint,Claude/claude-sonnet-4-6 - sent,Fietje/fietje-2-instruct - text,Fietje/fietje-2-instruct - lint,Fietje/fietje-2-instruct - sent
0,Oudegracht vormt het gezellige centrum van de ...,47,"38, 45, 61",De Oudegracht is het sfeervolle hart van de st...,19,"18, 32, 12",De Oudegracht is het sfeervolle hart van de st...,31,"18, 22, 44",Het klinkt als een prachtige plek. De gracht l...,31,"28, 46, 18, 8, 48"


In [273]:
b[0]['full_text'] = 'dikke vette huts'

In [274]:
b[0]

{'text': [{'original_sentence': 'De Oudegracht is het sfeervolle hart van de stad.',
   'rewritten_sentences': 'De Oudegracht vormt een levendige kern van de stad. Vroeger was dit waterweg een centrum vol handel met allerlei goederen die hier dagelijks werden aangevoerd en verplaatst. Tegenwoordig biedt het gebied een uitmuntend decor voor winkelbezoekers en bewoners om te genieten van lekkernijen of een maaltijd in de nabije oude stadspaleizen.'},
  {'original_sentence': 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.',
   'rewritten_sentences': 'In haar gloriedagen fungeerde de Oudegracht als een spilpunt in de lokale economie, waar handelaren opvallend actief waren. Echter, tegenwoordig genieten bezoekers van het gebied vooral van de charme en gezelligheid die dit historische element biedt.'}],
 'full_text': 'dikke vette huts'}

In [ ]:
# Volledige lint analyse hierover
" ".join([k['rewritten_sentences'] for k in b[0]['text']])

'De Oudegracht vormt een levendige kern van de stad. Vroeger was dit waterweg een centrum vol handel met allerlei goederen die hier dagelijks werden aangevoerd en verplaatst. Tegenwoordig biedt het gebied een uitmuntend decor voor winkelbezoekers en bewoners om te genieten van lekkernijen of een maaltijd in de nabije oude stadspaleizen. In haar gloriedagen fungeerde de Oudegracht als een spilpunt in de lokale economie, waar handelaren opvallend actief waren. Echter, tegenwoordig genieten bezoekers van het gebied vooral van de charme en gezelligheid die dit historische element biedt.'

In [95]:
rep = [inp_txt, 'De Oudegracht is het mooie hart van de stad.  \nIn de middeleeuwen was het hier druk met het verplaatsen van spullen.  \nNu is het een fijne plek om te winkelen en te eten in oude kastelen.',
 'De Oudegracht is het sfeervolle hart van de stad.\nVroeger was de gracht erg druk. Boten brachten toen veel spullen naar de stad.\nNu kun je er goed eten in een kasteel. Je kunt er ook spullen kopen in de winkel.',
 'De Oudegracht is het sfeervolle hart van de stad.\nIn de middeleeuwen was het hier druk met het laden en lossen van goederen.\nNu is het een mooie plek om te winkelen. Je kan er ook lunchen of eten in de oude stadskastelen.',
 'De Oudegracht vormde in de middeleeuwen inderdaad het economisch kloppend hart van Utrecht, door de handel en het transport van goederen. Deze periode was gekenmerkt door levendige markten en een rijk sociaal leven langs de grachten. Nu biedt de Oudegracht een scala aan winkelmogelijkheden, gezellige cafés en restaurants die de sfeer van het historische stadscentrum vastleggen, met name in de charmante oude stadspaleizen en kastelen.']

In [97]:
for r in rep:
    print(r)
    anal = ReadabilityAnalysis.from_text(r)
    print(anal.lint_scores_per_sentence, " === ", anal.lint.score)

De Oudegracht is het sfeervolle hart van de stad.
In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. 
Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.
[18.511612982419507, 54.27056340066443, 63.24402181810589]  ===  48.20593518603563
De Oudegracht is het mooie hart van de stad.  
In de middeleeuwen was het hier druk met het verplaatsen van spullen.  
Nu is het een fijne plek om te winkelen en te eten in oude kastelen.
[18.2557703209027, 25.36036942506128, 38.21288195313995]  ===  28.380658349881514
De Oudegracht is het sfeervolle hart van de stad.
Vroeger was de gracht erg druk. Boten brachten toen veel spullen naar de stad.
Nu kun je er goed eten in een kasteel. Je kunt er ook spullen kopen in de winkel.
[18.511612982419507, 20.92452028300255, 25.755641634770996, 6.486819525985041, 16.050486276907378]  ===  18.7211031833437
De Oudegracht is het sfeervolle hart van de stad.
In de middeleeuwen was he

In [123]:
fietje = RE_LMStudio(
    model="fietje-2-instruct",
    name="Fietje",
    is_local=True,
    is_open_source=True,
    is_large=False
)

In [124]:
fietje.set_instruction(instruction_nl)

In [125]:
fietje.prompt_model(inp_txt)

{'text': [{'original_sentence': 'De Oudegracht is het sfeervolle hart van de stad.',
   'rewritten_sentences': 'De Oudegracht vormt het levendige middelpunt van de stad.'},
  {'original_sentence': 'In de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen.',
   'rewritten_sentences': 'Tijdens de middeleeuwen was dit gebied een levendig centrum waar handel plaatsvond.'},
  {'original_sentence': 'Nu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.',
   'rewritten_sentences': 'Vandaag de dag biedt de Oudegracht een schitterend decor voor winkelbezoek, maar het is ook een ideaal uitgangspunt voor een lunch of diner in de historische stadskastelen.'}]}

In [207]:
inp_txt

'De Oudegracht is het sfeervolle hart van de stad.\nIn de middeleeuwen was het hier een drukte van belang met de aan- en afvoer van goederen. \nNu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'

In [15]:
k = 'De Oudegracht is het sfeervolle hart van de stad.\nIn de middeleeuwen was het hier een drukte van belang met de <text_eval>aan- en afvoer<text_eval> van goederen. \nNu is het een prachtige plek om te winkelen en te lunchen of te dineren in de oude stadskastelen.'

In [81]:
# claude = RE_Claude(
#     model='claude-sonnet-4-6',
#     name="Claude",
#     is_local=False,
#     is_open_source=False,
#     is_large=False
# )
claude.set_instruction("Rewrite the text so that it is easier to understand. Only change text between <text_eval> tags.")
bf = claude.prompt_model(k)

In [82]:
bf

RewrittenText(text=[RewriteResponse(original_sentence='aan- en afvoer', rewritten_sentences='aanvoer en afvoer')])

In [98]:
llk = [1, 2, 3, 4]

for i in llk:
    if i < 4:
        continue
        print("dont print this")
    print("Element is groter dan of gelijk aan 4")

Element is groter dan of gelijk aan 4
